# SymbolicModel Demo

In this demo we show you how to:
* Use `SymbolicModel` with any callable function (model-agnostic)
* Perform symbolic regression with the `.fit()` method
* Make predictions with the `.predict()` method
* Access equations with the `.get_equation()` method

## Create a PyTorch Model

We'll create a simple PyTorch model and train it. SymbolicModel can work with any callable, including PyTorch models.

In [1]:
import torch
import numpy as np
import torch.nn as nn

class MLP(nn.Module):
    """
    Simple MLP.
    """
    def __init__(self, input_dim, output_dim, hidden_dim):
        super(MLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.mlp(x)

class SimpleModel(nn.Module):
    """
    Simple model class.
    """
    def __init__(self, input_dim, output_dim, hidden_dim = 64):
        super(SimpleModel, self).__init__()

        self.mlp = MLP(input_dim, output_dim, hidden_dim)

    def forward(self, x):
        x = self.mlp(x)
        return x

Train the model on some data.

$$
y = x_0^2 +3 \sin(x_4)-4
$$

In [2]:
# Make the dataset 
x = np.array([np.random.uniform(0, 1, 10_000) for _ in range(5)]).T  
y = x[:, 0]**2 + 3*np.sin(x[:, 4]) - 4
noise = np.array([np.random.normal(0, 0.05*np.std(y)) for _ in range(len(y))])
y = y + noise 

In [3]:
# Set up training

import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

def train_model(model, dataloader, opt, criterion, epochs = 100):
    """
    Train a model for the specified number of epochs.
    
    Args:
        model: PyTorch model to train
        dataloader: DataLoader for training data
        opt: Optimizer
        criterion: Loss function
        epochs: Number of training epochs
        
    Returns:
        tuple: (trained_model, loss_tracker)
    """
    loss_tracker = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_x, batch_y in dataloader:
            # Forward pass
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            
            # Backward pass
            opt.zero_grad()
            loss.backward()
            opt.step()
            
            epoch_loss += loss.item()
        
        loss_tracker.append(epoch_loss)
        if (epoch + 1) % 5 == 0:
            avg_loss = epoch_loss / len(dataloader)
            print(f'Epoch [{epoch+1}/{epochs}], Avg Loss: {avg_loss:.6f}')
    return model, loss_tracker

# Instantiate the model
model = SimpleModel(input_dim=x.shape[1], output_dim=1)

# Set up training
criterion = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr=0.001)
X_train, X_test, y_train, y_test = train_test_split(x, y.reshape(-1,1), test_size=0.2, random_state=290402)

# Set up dataset
dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [4]:
# Train the model and save the weights

model, losses = train_model(model, dataloader, opt, criterion, 20)
torch.save(model.state_dict(), 'model_weights.pth')

Epoch [5/20], Avg Loss: 0.083512
Epoch [10/20], Avg Loss: 0.060172
Epoch [15/20], Avg Loss: 0.046609
Epoch [20/20], Avg Loss: 0.038767


## Use SymbolicModel with the model-agnostic API

Instead of wrapping the model, we define a callable function and pass it to `fit()`.

In [5]:
from symtorch import SymbolicModel

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [6]:
# Define a callable function that wraps the model
def f(inputs):
    """Black-box function for symbolic regression"""
    model.eval()
    with torch.no_grad():
        # Convert numpy to torch if needed
        if isinstance(inputs, np.ndarray):
            inputs = torch.FloatTensor(inputs)
        return model(inputs)

# Create SymbolicModel instance
symbolic_model = SymbolicModel()

## Perform symbolic regression using `.fit()`

The new API uses `.fit(f, inputs)` instead of `.distill(inputs)`.

In [7]:
# Fit symbolic regression
symbolic_model.fit(f, torch.FloatTensor(X_test))

/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...


Running SR on output dimension 0 of 0


[ Info: Started!



Expressions evaluated per second: 3.790e+05
Progress: 2240 / 12400 total iterations (18.065%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.699e-01  0.000e+00  y = -2.2732
3           2.470e-01  4.182e-01  y = x₄ + -2.7734
5           8.157e-02  5.539e-01  y = (x₄ * 2.3919) + -3.4697
7           8.876e-03  1.109e+00  y = ((x₄ * 2.3873) + -3.9596) + x₀
9           6.655e-03  1.440e-01  y = ((x₄ * 2.3803) + (x₀ * x₀)) + -3.7907
10          4.830e-03  3.206e-01  y = (sin(x₄) * 2.7931) + (x₀ + -4.0487)
12          2.412e-03  3.472e-01  y = ((sin(x₄) * 2.7854) + -3.8798) + (x₀ * x₀)
14          1.745e-03  1.618e-01  y = ((sin(x₄) * 2.7868) + (x₀ * (x₀ * 0.91376))) + -3.8522
15          1.679e-03  3.847e-02  y = sin(x₀ * x₀) + ((sin(x₄) * 2.7881) + -3.8583)
16          1.477e-03  1.

[ Info: Final population:
[ Info: Results saved to:


  - SR_output/SymbolicModel/dim0_1763552257/hall_of_fame.csv


## Access the discovered symbolic equation

Use `.get_equation()` to see the discovered equation.

In [9]:
# Get the best equation
equation = symbolic_model.get_equation()
print(f"Discovered equation: {equation}")

# You can also access equations for multi-output models
# equation_output_0 = symbolic_model.get_equation(output_idx=0)

# Or get a simpler equation at a specific complexity level
simple_equation = symbolic_model.get_equation(complexity=14)
print(simple_equation)

Discovered equation: (sin(x4 + 1.2555817) * ((x0 * x0) + ((x4 + -0.3380143) * 2.7570567))) + -2.9624221
((sin(x4) * 2.7867591) + (x0 * (x0 * 0.9137458))) + -3.852188


## Make predictions with the symbolic equation

Use `.predict()` to make predictions with the discovered symbolic equation.

In [10]:
# Make predictions using the symbolic equation
predictions = symbolic_model.predict(X_test)
print(f"Predictions shape: {predictions.shape}")

# Compare with original model
original_predictions = f(X_test).numpy()
mse = np.mean((predictions - original_predictions.flatten())**2)
print(f"MSE between symbolic and original model: {mse:.6f}")

Predictions shape: (2000,)
MSE between symbolic and original model: 0.001138


## Example: Model-agnostic usage with pure Python function

SymbolicModel works with ANY callable function, not just PyTorch models!

In [11]:
# Define a pure Python/NumPy function
def my_function(x):
    """A simple mathematical function: x0^2 + 3*sin(x4) - 4"""
    return x[:, 0]**2 + 3*np.sin(x[:, 4]) - 4

# Generate test data
X_func_test = np.random.uniform(0, 1, (100, 5))

# Fit symbolic regression directly to the function
symbolic_func = SymbolicModel(sr_params={'niterations': 200})
symbolic_func.fit(my_function, X_func_test)

# Get the equation
discovered_eq = symbolic_func.get_equation()
print(f"Original function: x0^2 + 3*sin(x4) - 4")
print(f"Discovered equation: {discovered_eq}")

# Make predictions
func_predictions = symbolic_func.predict(X_func_test)
true_values = my_function(X_func_test)
mse = np.mean((func_predictions - true_values)**2)
print(f"MSE: {mse:.8f}")

Running SR on output dimension 0 of 0


/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           7.453e-01  0.000e+00  y = -2.2744
3           3.440e-01  3.866e-01  y = x₄ + -2.7814
5           7.904e-02  7.354e-01  y = (x₄ * 2.7073) + -3.6471
7           8.500e-03  1.115e+00  y = (x₄ * 2.5718) + (x₀ + -4.0834)
9           2.720e-03  5.698e-01  y = (x₀ * x₀) + ((x₄ * 2.5666) + -3.9095)
11          2.702e-03  3.311e-03  y = (x₄ * 2.5687) + (((x₀ + -0.015278) * x₀) + -3.9028)
12          1.297e-14  2.606e+01  y = (x₀ * x₀) + ((sin(x₄) * 3) + -4)
14          9.628e-15  1.489e-01  y = (((sin(x₄) + -0.46359) * 3) + (x₀ * x₀)) + -2.6092
19          8.207e-15  3.194e-02  y = (((sin(x₄) + (x₀ * x₀)) + -1.1619) + (sin(x₄) * 2)) + ...
                                      -2.8381
23          8.065e-15  4.367e-03  y = ((sin(x₄ + ((x₀ * x₀) * -3.648e-08)) + (x₀ * x₀)) + (s...
                                      in(x₄) * 2)) + -4
25       

In [12]:
# Clean up 

import os
import shutil
if os.path.exists('SR_output'):
    shutil.rmtree('SR_output')
    os.remove('model_weights.pth')